# CDT-III 2-Stage VCE: CD52/Alemtuzumab Case Study (NB3)
## Drug Efficacy, Side Effects, Resistance — Integrated Prediction

**Purpose**: Comprehensive CD52 knockdown analysis as Alemtuzumab model.
RNA + Protein simultaneous prediction, side effect map, resistance mechanisms.

**CDT-III Unique**: Only CDT-III can predict protein-level drug effects.
CDT-II sees mRNA changes only; CDT-III reveals the full translational picture.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
from typing import Dict, Optional, Tuple, List
import numpy as np
import h5py
import pandas as pd
from pathlib import Path
from scipy.stats import pearsonr, spearmanr
import matplotlib.pyplot as plt
import seaborn as sns
import json
import gc

print(f"PyTorch: {torch.__version__}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# For t-SNE in VCE embedding analysis
try:
    from sklearn.manifold import TSNE
    print('sklearn available')
except ImportError:
    print('sklearn not found, install with: pip install scikit-learn')

In [ ]:
# ============================================================
# Path Configuration
# ============================================================
DRIVE_BASE = Path("/content/drive/MyDrive/cdt_data")
OUTPUT_BASE = Path("/content/drive/MyDrive/cdt_outputs/morris_2stage_vce_v2")

CELLLEVEL_WITH_ADT_PATH = DRIVE_BASE / "morris_celllevel_effects_with_adt.h5"
TSS_ENFORMER_PATH = DRIVE_BASE / "morris_28genes_enformer.h5"
BEST_MODEL_PATH = OUTPUT_BASE / "cdt_2stage_vce_best.pt"
ADT_MAPPING_PATH = "/content/drive/MyDrive/cdt_data/adt_mapping.csv"

FIGURES_DIR = OUTPUT_BASE / "cd52_alemtuzumab_figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

for name, path in [("Cell-level + ADT", CELLLEVEL_WITH_ADT_PATH),
                    ("TSS Enformer", TSS_ENFORMER_PATH),
                    ("Best model", BEST_MODEL_PATH)]:
    print(f"  {'✓' if path.exists() else '✗'} {name}: {path}")

In [ ]:
# ============================================================
# CDT-III 2-Stage VCE Model Definition
# (Copy from CDT_Morris_2StageVCE_v2_Training.ipynb cells 12-16)
# ============================================================

@dataclass
class CDT2StageVCEConfig:
    dna_dim: int = 3072
    dna_seq_len: int = 896
    n_genes: int = 2361
    n_proteins: int = 189
    hidden_dim: int = 512
    nhead: int = 8
    dropout: float = 0.3
    protein_dropout: float = 0.5
    dna_self_attn_layers: int = 2
    rna_self_attn_layers: int = 1
    protein_self_attn_layers: int = 1


class RawExpressionEncoder(nn.Module):
    def __init__(self, n_features, hidden_dim, dropout=0.1):
        super().__init__()
        self.n_features = n_features
        self.hidden_dim = hidden_dim
        self.feature_embedding = nn.Embedding(n_features, hidden_dim)
        self.expr_projector = nn.Sequential(
            nn.Linear(1, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout))
        self.combine = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.Dropout(dropout))

    def forward(self, expression):
        B = expression.size(0)
        feat_ids = torch.arange(self.n_features, device=expression.device)
        feat_emb = self.feature_embedding(feat_ids).unsqueeze(0).expand(B, -1, -1)
        expr_emb = self.expr_projector(expression.unsqueeze(-1))
        return self.combine(torch.cat([feat_emb, expr_emb], dim=-1))


class SequenceProjector(nn.Module):
    def __init__(self, input_dim, output_dim, dropout=0.1):
        super().__init__()
        self.linear = nn.Linear(input_dim, output_dim)
        self.norm = nn.LayerNorm(output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.norm(self.linear(x)))


class FlashSelfAttentionBlock(nn.Module):
    def __init__(self, d_model, nhead=8, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.nhead = nhead
        self.head_dim = d_model // nhead
        self.dropout_p = dropout
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model), nn.Dropout(dropout))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, return_attn=False):
        B, S, _ = x.shape
        Q = self.q_proj(x).view(B, S, self.nhead, self.head_dim).transpose(1, 2)
        K = self.k_proj(x).view(B, S, self.nhead, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).view(B, S, self.nhead, self.head_dim).transpose(1, 2)
        if return_attn:
            scale = self.head_dim ** -0.5
            attn_weights = torch.matmul(Q, K.transpose(-2, -1)) * scale
            attn_weights = F.softmax(attn_weights, dim=-1)
            attn_out = torch.matmul(attn_weights, V)
        else:
            attn_out = F.scaled_dot_product_attention(
                Q, K, V, dropout_p=self.dropout_p if self.training else 0.0)
            attn_weights = None
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, S, self.d_model)
        attn_out = self.out_proj(attn_out)
        x = self.norm1(x + self.dropout(attn_out))
        x = self.norm2(x + self.ffn(x))
        if return_attn:
            return x, attn_weights
        return x


class FlashCrossAttentionBlock(nn.Module):
    def __init__(self, d_model, nhead=8, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.nhead = nhead
        self.head_dim = d_model // nhead
        self.dropout_p = dropout
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model), nn.Dropout(dropout))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key_value, return_attn=False):
        B, QL, _ = query.shape
        KL = key_value.shape[1]
        Q = self.q_proj(query).view(B, QL, self.nhead, self.head_dim).transpose(1, 2)
        K = self.k_proj(key_value).view(B, KL, self.nhead, self.head_dim).transpose(1, 2)
        V = self.v_proj(key_value).view(B, KL, self.nhead, self.head_dim).transpose(1, 2)
        if return_attn:
            scale = self.head_dim ** -0.5
            attn_weights = torch.matmul(Q, K.transpose(-2, -1)) * scale
            attn_weights = F.softmax(attn_weights, dim=-1)
            attn_out = torch.matmul(attn_weights, V)
        else:
            attn_out = F.scaled_dot_product_attention(
                Q, K, V, dropout_p=self.dropout_p if self.training else 0.0)
            attn_weights = None
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, QL, self.d_model)
        attn_out = self.out_proj(attn_out)
        x = self.norm1(query + self.dropout(attn_out))
        x = self.norm2(x + self.ffn(x))
        if return_attn:
            return x, attn_weights
        return x


class VirtualCellEmbedderDNARNA(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.nhead = 4
        self.head_dim = d_model // self.nhead
        self.dna_query = nn.Parameter(torch.randn(1, 1, d_model))
        self.rna_query = nn.Parameter(torch.randn(1, 1, d_model))
        self.dna_q_proj = nn.Linear(d_model, d_model)
        self.dna_k_proj = nn.Linear(d_model, d_model)
        self.dna_v_proj = nn.Linear(d_model, d_model)
        self.dna_out_proj = nn.Linear(d_model, d_model)
        self.rna_q_proj = nn.Linear(d_model, d_model)
        self.rna_k_proj = nn.Linear(d_model, d_model)
        self.rna_v_proj = nn.Linear(d_model, d_model)
        self.rna_out_proj = nn.Linear(d_model, d_model)
        self.fusion = nn.Sequential(
            nn.Linear(d_model * 2, d_model * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model), nn.LayerNorm(d_model))

    def _attention_pool(self, query, key_value, q_proj, k_proj, v_proj, out_proj):
        B, S = key_value.size(0), key_value.size(1)
        query = query.expand(B, -1, -1)
        Q = q_proj(query).view(B, 1, self.nhead, self.head_dim).transpose(1, 2)
        K = k_proj(key_value).view(B, S, self.nhead, self.head_dim).transpose(1, 2)
        V = v_proj(key_value).view(B, S, self.nhead, self.head_dim).transpose(1, 2)
        attn_out = F.scaled_dot_product_attention(Q, K, V)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, 1, self.d_model)
        return out_proj(attn_out).squeeze(1)

    def forward(self, dna_encoded, rna_encoded):
        dna_pooled = self._attention_pool(
            self.dna_query, dna_encoded,
            self.dna_q_proj, self.dna_k_proj, self.dna_v_proj, self.dna_out_proj)
        rna_pooled = self._attention_pool(
            self.rna_query, rna_encoded,
            self.rna_q_proj, self.rna_k_proj, self.rna_v_proj, self.rna_out_proj)
        return self.fusion(torch.cat([dna_pooled, rna_pooled], dim=-1))


class VirtualCellEmbedderProtein(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.nhead = 4
        self.head_dim = d_model // self.nhead
        self.protein_query = nn.Parameter(torch.randn(1, 1, d_model))
        self.protein_q_proj = nn.Linear(d_model, d_model)
        self.protein_k_proj = nn.Linear(d_model, d_model)
        self.protein_v_proj = nn.Linear(d_model, d_model)
        self.protein_out_proj = nn.Linear(d_model, d_model)
        self.fusion = nn.Sequential(
            nn.Linear(d_model * 2, d_model * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model), nn.LayerNorm(d_model))

    def _attention_pool(self, query, key_value, q_proj, k_proj, v_proj, out_proj):
        B, S = key_value.size(0), key_value.size(1)
        query = query.expand(B, -1, -1)
        Q = q_proj(query).view(B, 1, self.nhead, self.head_dim).transpose(1, 2)
        K = k_proj(key_value).view(B, S, self.nhead, self.head_dim).transpose(1, 2)
        V = v_proj(key_value).view(B, S, self.nhead, self.head_dim).transpose(1, 2)
        attn_out = F.scaled_dot_product_attention(Q, K, V)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, 1, self.d_model)
        return out_proj(attn_out).squeeze(1)

    def forward(self, cell_emb_rna, protein_encoded):
        protein_pooled = self._attention_pool(
            self.protein_query, protein_encoded,
            self.protein_q_proj, self.protein_k_proj,
            self.protein_v_proj, self.protein_out_proj)
        return self.fusion(torch.cat([cell_emb_rna, protein_pooled], dim=-1))


class CDTTrimodal2StageModel(nn.Module):
    def __init__(self, config=None):
        super().__init__()
        if config is None:
            config = CDT2StageVCEConfig()
        self.config = config
        self.dna_projector = SequenceProjector(config.dna_dim, config.hidden_dim, config.dropout)
        self.dna_self_attn_layers = nn.ModuleList([
            FlashSelfAttentionBlock(config.hidden_dim, config.nhead, config.dropout)
            for _ in range(config.dna_self_attn_layers)])
        self.rna_encoder = RawExpressionEncoder(config.n_genes, config.hidden_dim, config.dropout)
        self.rna_self_attn_layers = nn.ModuleList([
            FlashSelfAttentionBlock(config.hidden_dim, config.nhead, config.dropout)
            for _ in range(config.rna_self_attn_layers)])
        self.dna_to_rna = FlashCrossAttentionBlock(config.hidden_dim, config.nhead, config.dropout)
        self.vce_t = VirtualCellEmbedderDNARNA(config.hidden_dim, config.dropout)
        self.rna_task_layer = nn.Sequential(
            nn.Linear(config.hidden_dim, config.hidden_dim), nn.GELU(),
            nn.Dropout(config.dropout), nn.Linear(config.hidden_dim, config.n_genes))
        self.protein_encoder = RawExpressionEncoder(
            config.n_proteins, config.hidden_dim, config.protein_dropout)
        self.protein_self_attn_layers = nn.ModuleList([
            FlashSelfAttentionBlock(config.hidden_dim, config.nhead, config.protein_dropout)
            for _ in range(config.protein_self_attn_layers)])
        self.rna_to_protein = FlashCrossAttentionBlock(
            config.hidden_dim, config.nhead, config.protein_dropout)
        self.vce_p = VirtualCellEmbedderProtein(config.hidden_dim, config.protein_dropout)
        self.protein_task_layer = nn.Sequential(
            nn.Linear(config.hidden_dim, config.hidden_dim), nn.GELU(),
            nn.Dropout(config.protein_dropout),
            nn.Linear(config.hidden_dim, config.n_proteins))

    def forward(self, dna_emb, rna_expr, protein_expr, return_attention=False):
        attn_maps = {} if return_attention else None
        dna = self.dna_projector(dna_emb)
        rna = self.rna_encoder(rna_expr)
        protein = self.protein_encoder(protein_expr)
        for i, layer in enumerate(self.dna_self_attn_layers):
            if return_attention:
                dna, attn_w = layer(dna, return_attn=True)
                attn_maps[f'dna_self_attn_{i}'] = attn_w.detach().cpu()
            else:
                dna = layer(dna)
        for i, layer in enumerate(self.rna_self_attn_layers):
            if return_attention:
                rna, attn_w = layer(rna, return_attn=True)
                attn_maps[f'rna_self_attn_{i}'] = attn_w.detach().cpu()
            else:
                rna = layer(rna)
        for i, layer in enumerate(self.protein_self_attn_layers):
            if return_attention:
                protein, attn_w = layer(protein, return_attn=True)
                attn_maps[f'protein_self_attn_{i}'] = attn_w.detach().cpu()
            else:
                protein = layer(protein)
        if return_attention:
            rna, attn_w = self.dna_to_rna(query=rna, key_value=dna, return_attn=True)
            attn_maps['dna_to_rna_cross'] = attn_w.detach().cpu()
        else:
            rna = self.dna_to_rna(query=rna, key_value=dna)
        if return_attention:
            protein, attn_w = self.rna_to_protein(query=protein, key_value=rna, return_attn=True)
            attn_maps['rna_to_protein_cross'] = attn_w.detach().cpu()
        else:
            protein = self.rna_to_protein(query=protein, key_value=rna)
        cell_emb_rna = self.vce_t(dna, rna)
        rna_pred = self.rna_task_layer(cell_emb_rna)
        cell_emb_protein = self.vce_p(cell_emb_rna, protein)
        protein_pred = self.protein_task_layer(cell_emb_protein)
        if return_attention:
            return rna_pred, protein_pred, attn_maps
        return rna_pred, protein_pred

    def get_num_params(self, trainable_only=False):
        if trainable_only:
            return sum(p.numel() for p in self.parameters() if p.requires_grad)
        return sum(p.numel() for p in self.parameters())

print("CDTTrimodal2StageModel defined.")

In [ ]:
# ============================================================
# Load Data
# ============================================================
print("Loading integrated RNA + ADT data...")
with h5py.File(CELLLEVEL_WITH_ADT_PATH, 'r') as f:
    rna_log2fc = f['log2fc'][:]
    cell_expr = f['cell_expr'][:]
    target_gene_idx = f['target_gene_idx'][:]
    cdt_genes = [g.decode() if isinstance(g, bytes) else g for g in f['gene_names_cdt'][:]]
    target_gene_names = [g.decode() if isinstance(g, bytes) else g for g in f['target_gene_names'][:]]
    ntc_mean_expr = f['ntc_mean_expr'][:]
    train_genes = [g.decode() if isinstance(g, bytes) else g for g in f['train_genes'][:]]
    val_genes = [g.decode() if isinstance(g, bytes) else g for g in f['val_genes'][:]]
    protein_effect = f['protein_effect'][:]
    protein_dsb = f['protein_dsb'][:]
    protein_names = [g.decode() if isinstance(g, bytes) else g for g in f['protein_names'][:]]
    n_cells = f.attrs['n_cells']
    n_genes = f.attrs['n_genes']
    n_proteins = f.attrs['n_proteins']

    # Check if 'protein_matched_mask' exists, otherwise create a default all-True mask
    if 'protein_matched_mask' in f:
        protein_matched_mask = f['protein_matched_mask'][:]
    else:
        print("Warning: 'protein_matched_mask' not found in HDF5. Creating an all-True mask.")
        protein_matched_mask = np.ones(n_cells, dtype=bool)

print(f"RNA: {n_cells} cells, {n_genes} genes, log2FC {rna_log2fc.shape}")
print(f"Protein: {n_proteins} proteins, effect {protein_effect.shape}")
print(f"Cells with protein: {protein_matched_mask.sum()}/{n_cells}")
print(f"Train genes ({len(train_genes)}): {train_genes[:5]}...")
print(f"Val genes ({len(val_genes)}): {val_genes}")

# Enformer
print("\nLoading Enformer embeddings...")
with h5py.File(TSS_ENFORMER_PATH, 'r') as f:
    enformer_emb = f['embeddings'][:]
    enformer_genes = [g.decode() if isinstance(g, bytes) else g for g in f['gene_names'][:]]
print(f"Enformer: {enformer_emb.shape}")

target_to_enformer = {}
for i, gene in enumerate(target_gene_names):
    if gene in enformer_genes:
        target_to_enformer[i] = enformer_genes.index(gene)
print(f"TSS matched: {len(target_to_enformer)}/{len(target_gene_names)}")

# ADT mapping
try:
    adt_mapping = pd.read_csv(ADT_MAPPING_PATH)
    adt_mapping = adt_mapping[~adt_mapping['morris_adt_id'].str.startswith('ADT-CTL')]
    adt_to_gene = dict(zip(adt_mapping['morris_adt_id'], adt_mapping['gene_symbol']))
    adt_to_protein = dict(zip(adt_mapping['morris_adt_id'], adt_mapping['protein_name']))
    print(f"ADT mapping: {len(adt_to_gene)} proteins mapped")
except:
    print("ADT mapping not found, using protein_names directly")
    adt_to_gene = {}
    adt_to_protein = {}

# Index maps
gene_to_cdt_idx = {g: i for i, g in enumerate(cdt_genes)}
gene_name_to_target_idx = {n: i for i, n in enumerate(target_gene_names)}

# ADT screening: expressed proteins
mean_dsb = protein_dsb.mean(axis=0)
std_dsb = protein_dsb.std(axis=0)
expressed_mask = (mean_dsb > 0.5) & (std_dsb > 0.1)
expressed_indices = np.where(expressed_mask)[0]
n_expressed = expressed_mask.sum()
print(f"\nExpressed proteins: {n_expressed}/{n_proteins}")

In [ ]:
import os
output_dir = "/content/drive/MyDrive/cdt_outputs/morris_2stage_vce_v2/"
if os.path.exists(output_dir):
    for f in sorted(os.listdir(output_dir)):
        size = os.path.getsize(os.path.join(output_dir, f))
        print(f"  {f} ({size/1e6:.1f} MB)")
else:
    print("Directory not found!")
    # Check parent
    parent = "/content/drive/MyDrive/cdt_outputs/"
    if os.path.exists(parent):
        print("Available dirs:")
        for d in sorted(os.listdir(parent)):
            print(f"  {d}")


In [ ]:
# ============================================================
# Load Best Model Checkpoint
# ============================================================
print("Loading best model checkpoint...")
config = CDT2StageVCEConfig(n_genes=len(cdt_genes), n_proteins=len(protein_names))
model = CDTTrimodal2StageModel(config).to(device)

ckpt = torch.load(BEST_MODEL_PATH, map_location=device, weights_only=False)
if 'model_state_dict' in ckpt:
    model.load_state_dict(ckpt['model_state_dict'])
    print(f"Loaded from checkpoint with keys: {list(ckpt.keys())}")
    if 'val_rna_r' in ckpt:
        print(f"  Val RNA r: {ckpt['val_rna_r']:.4f}")
    if 'val_protein_r' in ckpt:
        print(f"  Val Protein r: {ckpt['val_protein_r']:.4f}")
    if 'epoch' in ckpt:
        print(f"  Epoch: {ckpt['epoch']}")
else:
    model.load_state_dict(ckpt)
    print("Loaded raw state dict")

model.eval()
print(f"Parameters: {model.get_num_params():,}")

In [ ]:
# ============================================================
# A. CD52 / Alemtuzumab Setup
# ============================================================
# CD52: GPI-anchored glycoprotein on lymphocyte surface
# Alemtuzumab (Lemtrada): Anti-CD52 antibody
#   - Approved: CLL, MS, transplant conditioning
#   - Known side effects: autoimmune thyroiditis (30-40%), ITP (2-3%),
#     infection risk, autoimmune nephropathy

CD52_TARGET_IDX = gene_name_to_target_idx.get('CD52')
CD52_CDT_IDX = gene_to_cdt_idx.get('CD52')

# Find CD52 ADT protein
CD52_ADT_IDX = None
for i, pname in enumerate(protein_names):
    if adt_to_gene.get(pname) == 'CD52' or 'CD52' in adt_to_protein.get(pname, ''):
        CD52_ADT_IDX = i
        break

print(f"CD52 indices:")
print(f"  Target gene idx: {CD52_TARGET_IDX}")
print(f"  CDT gene idx: {CD52_CDT_IDX}")
print(f"  ADT protein idx: {CD52_ADT_IDX}")
if CD52_ADT_IDX is not None:
    print(f"  ADT name: {protein_names[CD52_ADT_IDX]}")
    print(f"  Protein: {adt_to_protein.get(protein_names[CD52_ADT_IDX], '?')}")

# Get CD52 KD cells
cd52_cells = [i for i in range(n_cells)
              if target_gene_idx[i] == CD52_TARGET_IDX
              and CD52_TARGET_IDX in target_to_enformer
              and protein_matched_mask[i]]
print(f"\nCD52 KD cells with protein data: {len(cd52_cells)}")

In [ ]:
# ============================================================
# B. CD52 KD → RNA + Protein Simultaneous Prediction
# ============================================================

if CD52_TARGET_IDX is None or CD52_TARGET_IDX not in target_to_enformer:
    raise RuntimeError('CD52 not found in target genes or Enformer. Cannot proceed.')

all_rna_preds, all_rna_targets = [], []
all_prot_preds, all_prot_targets = [], []

enf_idx = target_to_enformer[CD52_TARGET_IDX]

for ci in cd52_cells:
    dna_t = torch.from_numpy(enformer_emb[enf_idx:enf_idx+1].astype(np.float32)).to(device)
    rna_t = torch.from_numpy(cell_expr[ci:ci+1].astype(np.float32)).to(device)
    prot_t = torch.from_numpy(protein_dsb[ci:ci+1].astype(np.float32)).to(device)

    with torch.no_grad():
        rna_pred, prot_pred = model(dna_t, rna_t, prot_t)

    all_rna_preds.append(rna_pred.cpu().numpy().flatten())
    all_rna_targets.append(rna_log2fc[ci])
    all_prot_preds.append(prot_pred.cpu().numpy().flatten())
    all_prot_targets.append(protein_effect[ci])

all_rna_preds = np.array(all_rna_preds)
all_rna_targets = np.array(all_rna_targets)
all_prot_preds = np.array(all_prot_preds)
all_prot_targets = np.array(all_prot_targets)

mean_rna_pred = all_rna_preds.mean(axis=0)
mean_rna_target = all_rna_targets.mean(axis=0)
mean_prot_pred = all_prot_preds.mean(axis=0)
mean_prot_target = all_prot_targets.mean(axis=0)

rna_r = pearsonr(mean_rna_pred, mean_rna_target)[0]
prot_r = pearsonr(mean_prot_pred[expressed_mask], mean_prot_target[expressed_mask])[0]

print(f"CD52 KD Results ({len(cd52_cells)} cells):")
print(f"  RNA r(mean): {rna_r:.4f}")
print(f"  Protein r(mean, expressed): {prot_r:.4f}")

# Self-KD
if CD52_CDT_IDX is not None:
    self_kd_rna = pearsonr(all_rna_preds[:, CD52_CDT_IDX], all_rna_targets[:, CD52_CDT_IDX])[0]
    print(f"  Self-KD RNA (CD52→CD52 mRNA): r={self_kd_rna:.4f}")
    print(f"    Pred mean: {mean_rna_pred[CD52_CDT_IDX]:.4f}, Actual: {mean_rna_target[CD52_CDT_IDX]:.4f}")

if CD52_ADT_IDX is not None:
    self_kd_prot = pearsonr(all_prot_preds[:, CD52_ADT_IDX], all_prot_targets[:, CD52_ADT_IDX])[0]
    print(f"  Self-KD Protein (CD52→CD52 protein): r={self_kd_prot:.4f}")
    print(f"    Pred mean: {mean_prot_pred[CD52_ADT_IDX]:.4f}, Actual: {mean_prot_target[CD52_ADT_IDX]:.4f}")

In [ ]:
# ============================================================
# Figure 3 (NeurIPS): CD52/Alemtuzumab — 3 Panels
# ============================================================
# (A) CD52 KD RNA prediction vs actual (r = 0.748)
# (B) Protein prediction vs actual, 65 expressed (r = 0.962)
# (C) Side effect map: 25 proteins with detectable effects
#
# Outputs: cd52_panel_a.png, cd52_panel_b.png, cd52_panel_c.png (individual panels)

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import numpy as np

FIGURES_DIR = Path(output_dir) / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_FIG = Path("figures/neurips")
LOCAL_FIG.mkdir(parents=True, exist_ok=True)

# ---- ADT Mapping: always apply complete hardcoded mapping ----
# Morris et al. 2023 CITE-seq — 189 TotalSeq-B antibodies (from adt_mapping.csv)
# Format: ADT-ID -> (gene_symbol, protein_name)
_adt_map_hardcoded = {
    'ADT-0005': ('CD80', 'CD80'),
    'ADT-0007': ('CD274', 'CD274'),
    'ADT-0008': ('PDCD1LG2', 'CD273'),
    'ADT-0009': ('ICOSLG', 'CD275'),
    'ADT-0020': ('TNFRSF14', 'CD270'),
    'ADT-0021': ('TNFSF4', 'CD252'),
    'ADT-0022': ('TNFSF9', 'CD137L'),
    'ADT-0023': ('PVR', 'CD155'),
    'ADT-0024': ('NECTIN2', 'CD112'),
    'ADT-0026': ('CD47', 'CD47'),
    'ADT-0027': ('CD70', 'CD70'),
    'ADT-0028': ('TNFRSF8', 'CD30'),
    'ADT-0031': ('CD40', 'CD40'),
    'ADT-0032': ('CD40LG', 'CD154'),
    'ADT-0033': ('CD52', 'CD52'),
    'ADT-0034': ('CD3E', 'CD3'),
    'ADT-0046': ('CD8A', 'CD8'),
    'ADT-0047': ('NCAM1', 'CD56'),
    'ADT-0048': ('PTPRC', 'CD45'),
    'ADT-0050': ('CD19', 'CD19'),
    'ADT-0052': ('CD33', 'CD33'),
    'ADT-0054': ('CD34', 'CD34'),
    'ADT-0055': ('SDC1', 'CD138'),
    'ADT-0056': ('TNFRSF17', 'CD269'),
    'ADT-0057': ('B2M', 'β2-microglobulin'),
    'ADT-0058': ('HLA-A', 'HLA-ABC'),
    'ADT-0060': ('THY1', 'CD90'),
    'ADT-0061': ('KIT', 'CD117'),
    'ADT-0062': ('MME', 'CD10'),
    'ADT-0063': ('PTPRC', 'CD45RA'),
    'ADT-0064': ('IL3RA', 'CD123'),
    'ADT-0066': ('CD7', 'CD7'),
    'ADT-0070': ('ITGA6', 'CD49f'),
    'ADT-0071': ('CCR4', 'CD194'),
    'ADT-0072': ('CD4', 'CD4'),
    'ADT-0080': ('CD8A', 'CD8a'),
    'ADT-0081': ('CD14', 'CD14'),
    'ADT-0083': ('FCGR3A', 'CD16'),
    'ADT-0085': ('IL2RA', 'CD25'),
    'ADT-0087': ('PTPRC', 'CD45RO'),
    'ADT-0088': ('PDCD1', 'PD-1'),
    'ADT-0089': ('TIGIT', 'TIGIT'),
    'ADT-0100': ('MS4A1', 'CD20'),
    'ADT-0101': ('NCR1', 'CD335'),
    'ADT-0102': ('PTGDR2', 'CD294'),
    'ADT-0123': ('EPCAM', 'CD326'),
    'ADT-0124': ('PECAM1', 'CD31'),
    'ADT-0125': ('CD44', 'CD44'),
    'ADT-0127': ('PDPN', 'Podoplanin'),
    'ADT-0129': ('PDGFRB', 'CD140b'),
    'ADT-0134': ('MCAM', 'CD146 (MCAM)'),
    'ADT-0135': ('CDH1', 'CD324'),
    'ADT-0136': ('IGHM', 'IgM'),
    'ADT-0138': ('CD5', 'CD5'),
    'ADT-0139': ('TRGC1', 'TCR-γδ'),
    'ADT-0141': ('CCR5', 'CD195'),
    'ADT-0142': ('FCGR2A', 'CD32 (FcγRIIa)'),
    'ADT-0143': ('CCR6', 'CD196'),
    'ADT-0144': ('CXCR5', 'CD185'),
    'ADT-0145': ('ITGAE', 'CD103'),
    'ADT-0146': ('CD69', 'CD69'),
    'ADT-0147': ('SELL', 'CD62L'),
    'ADT-0149': ('KLRB1', 'CD161'),
    'ADT-0151': ('CTLA4', 'CTLA-4'),
    'ADT-0152': ('LAG3', 'LAG-3'),
    'ADT-0153': ('KLRG1', 'KLRG1'),
    'ADT-0154': ('CD27', 'CD27'),
    'ADT-0155': ('LAMP1', 'CD107a'),
    'ADT-0156': ('FAS', 'CD95 (Fas)'),
    'ADT-0158': ('TNFRSF4', 'CD134 (OX40)'),
    'ADT-0159': ('HLA-DRA', 'HLA-DR'),
    'ADT-0160': ('CD1C', 'CD1c'),
    'ADT-0161': ('ITGAM', 'CD11b'),
    'ADT-0162': ('FCGR1A', 'CD64'),
    'ADT-0163': ('THBD', 'CD141'),
    'ADT-0164': ('CD1D', 'CD1d'),
    'ADT-0165': ('KLRK1', 'CD314'),
    'ADT-0166': ('CEACAM8', 'CD66b'),
    'ADT-0167': ('CR1', 'CD35'),
    'ADT-0168': ('B3GAT1', 'CD57'),
    'ADT-0169': ('HAVCR2', 'TIM-3'),
    'ADT-0170': ('BTLA', 'CD272'),
    'ADT-0171': ('ICOS', 'CD278 (ICOS)'),
    'ADT-0174': ('CD58', 'CD58 (LFA-3)'),
    'ADT-0175': ('CD96', 'CD96'),
    'ADT-0176': ('ENTPD1', 'CD39'),
    'ADT-0177': ('FASLG', 'CD178'),
    'ADT-0179': ('CX3CR1', 'CX3CR1'),
    'ADT-0180': ('CD24', 'CD24'),
    'ADT-0181': ('CR2', 'CD21'),
    'ADT-0185': ('ITGAL', 'CD11a'),
    'ADT-0186': ('ITGAX', 'CD11c'),
    'ADT-0187': ('CD79B', 'CD79b'),
    'ADT-0188': ('CEACAM1', 'CD66a/c/e'),
    'ADT-0189': ('CD244', 'CD244'),
    'ADT-0196': ('GYPA', 'CD235ab'),
    'ADT-0205': ('MRC1', 'CD206'),
    'ADT-0206': ('SIGLEC1', 'CD169'),
    'ADT-0207': ('CLEC9A', 'CD370'),
    'ADT-0208': ('XCR1', 'XCR1'),
    'ADT-0214': ('ITGB7', 'Integrin-β7'),
    'ADT-0215': ('TNFRSF13C', 'BAFF-R'),
    'ADT-0217': ('ICAM1', 'CD54 (ICAM1)'),
    'ADT-0218': ('SELP', 'CD62P'),
    'ADT-0224': ('TRAC', 'TCR-αβ'),
    'ADT-0245': ('VCAM1', 'CD106'),
    'ADT-0246': ('IL2RB', 'CD122'),
    'ADT-0247': ('TNFRSF13B', 'CD267 (TACI)'),
    'ADT-0352': ('FCER1A', 'FcεRIα'),
    'ADT-0353': ('ITGA2B', 'CD41'),
    'ADT-0355': ('TNFRSF9', 'CD137 (4-1BB)'),
    'ADT-0356': ('TNFSF11', 'CD254'),
    'ADT-0358': ('CD163', 'CD163'),
    'ADT-0359': ('CD83', 'CD83'),
    'ADT-0360': ('TNFRSF18', 'CD357 (GITR)'),
    'ADT-0362': ('KDR', 'CD309 (VEGFR2)'),
    'ADT-0363': ('IL4R', 'CD124'),
    'ADT-0366': ('CXCR4', 'CD184 (CXCR4)'),
    'ADT-0367': ('CD2', 'CD2'),
    'ADT-0368': ('CD226', 'CD226 (DNAM-1)'),
    'ADT-0369': ('ITGB1', 'CD29'),
    'ADT-0370': ('CLEC4C', 'CD303'),
    'ADT-0371': ('ITGA2', 'CD49b'),
    'ADT-0373': ('CD81', 'CD81'),
    'ADT-0374': ('SLC3A2', 'CD98'),
    'ADT-0375': ('IGHG1', 'IgG-Fc'),
    'ADT-0384': ('IGHD', 'IgD'),
    'ADT-0385': ('ITGB2', 'CD18'),
    'ADT-0386': ('CD28', 'CD28'),
    'ADT-0387': ('CRLF2', 'TSLPR'),
    'ADT-0389': ('CD38', 'CD38'),
    'ADT-0390': ('IL7R', 'CD127'),
    'ADT-0392': ('FUT4', 'CD15'),
    'ADT-0393': ('CD22', 'CD22'),
    'ADT-0394': ('TFRC', 'CD71 (TFRC)'),
    'ADT-0395': ('VTCN1', 'B7-H4'),
    'ADT-0396': ('DPP4', 'CD26'),
    'ADT-0397': ('CCR3', 'CD193'),
    'ADT-0399': ('MSR1', 'CD204'),
    'ADT-0400': ('CDH5', 'CD144'),
    'ADT-0402': ('CD1A', 'CD1a'),
    'ADT-0406': ('NRP1', 'CD304'),
    'ADT-0407': ('CD36', 'CD36'),
    'ADT-0420': ('KIR2DL1', 'CD158'),
    'ADT-0437': ('CD207', 'CD207'),
    'ADT-0576': ('ITGA4', 'CD49d'),
    'ADT-0577': ('NT5E', 'CD73'),
    'ADT-0581': ('TRAV1-2', 'TCR-Vα7.2'),
    'ADT-0582': ('TRDV2', 'TCR-Vδ2'),
    'ADT-0583': ('TRGV9', 'TCR-Vγ9'),
    'ADT-0584': ('TRAV10', 'TCR-Vα24'),
    'ADT-0590': ('LAIR1', 'CD305 (LAIR1)'),
    'ADT-0591': ('OLR1', 'LOX-1'),
    'ADT-0592': ('KIR2DL2', 'CD158b'),
    'ADT-0594': ('PROM1', 'CD133'),
    'ADT-0597': ('CD209', 'CD209 (DC-SIGN)'),
    'ADT-0599': ('KIR3DL1', 'CD158e1'),
    'ADT-0600': ('KIR2DL5A', 'CD158f'),
    'ADT-0801': ('NCR1', 'CD337'),
    'ADT-0802': ('NCR2', 'CD336'),
    'ADT-0828': ('FCRL4', 'CD307d'),
    'ADT-0829': ('FCRL5', 'CD307e'),
    'ADT-0830': ('SLAMF7', 'CD319'),
    'ADT-0853': ('CLEC12A', 'CLEC12A'),
    'ADT-0863': ('TNFSF13B', 'BAFF'),
    'ADT-0867': ('KLRD1', 'CD94'),
    'ADT-0870': ('SLAMF1', 'CD150'),
    'ADT-0894': ('IGKC', 'Ig-κ'),
    'ADT-0895': ('LGALS3', 'Galectin-3'),
    'ADT-0896': ('LILRB1', 'CD85j'),
    'ADT-0897': ('FCER2', 'CD23'),
    'ADT-0898': ('IGLC1', 'Ig-λ'),
    'ADT-0899': ('HLA-A', 'HLA-A2'),
    'ADT-0900': ('CCR8', 'CD198'),
    'ADT-0901': ('LRRC32', 'GARP'),
    'ADT-0902': ('SIGLEC7', 'CD328'),
    'ADT-0908': ('TRBV6-5', 'TCR-Vβ13.1'),
    'ADT-0920': ('CD82', 'CD82'),
    'ADT-0944': ('CD101', 'CD101'),
    'ADT-0985': ('IL21R', 'CD360'),
    'ADT-1046': ('C5AR1', 'CD88'),
    'ADT-1047': ('HLA-F', 'HLA-F'),
    'ADT-1048': ('CD99', 'CD99'),
    'ADT-1049': ('FUT3', 'SEA-4'),
    'ADT-1051': ('PODXL', 'Podocalyxin'),
    'ADT-1052': ('GGT1', 'CD224'),
    'ADT-1056': ('TNFSF14', 'CD258'),
    'ADT-1057': ('TNFRSF25', 'DR3'),
    'ADT-5148': ('CLEC12A', 'CLL-1'),
}

# Always apply hardcoded mapping (overrides any partial/bad CSV mapping)
adt_to_gene = {}
adt_to_protein = {}
for pid in protein_names:
    if pid in _adt_map_hardcoded:
        gene, prot = _adt_map_hardcoded[pid]
        adt_to_gene[pid] = gene
        adt_to_protein[pid] = prot
    else:
        adt_to_protein[pid] = pid
        adt_to_gene[pid] = ''

n_mapped = sum(1 for v in adt_to_protein.values() if not v.startswith('ADT-'))
print(f"ADT mapping: {n_mapped}/{len(protein_names)} proteins mapped to names")

# Helper: get display name for a protein
def get_protein_label(pid):
    """Return readable protein label: 'Protein (Gene)' or 'Gene' if same."""
    prot = adt_to_protein.get(pid, pid)
    gene = adt_to_gene.get(pid, '')
    if prot.startswith('ADT-'):
        return gene if gene else prot
    if gene and gene not in prot:
        return f"{prot} ({gene})"
    return prot

# ---- Build side effect data (needed for panel C) ----
mean_prot_effect = mean_prot_target

expr_effects = []
for i in expressed_indices:
    pid = protein_names[i]
    expr_effects.append({
        'idx': i,
        'pid': pid,
        'label': get_protein_label(pid),
        'gene': adt_to_gene.get(pid, ''),
        'actual': mean_prot_effect[i],
        'predicted': mean_prot_pred[i],
    })
expr_effects.sort(key=lambda x: abs(x['actual']), reverse=True)

# Filter detectable (|effect| > 0.1)
detectable = [e for e in expr_effects if abs(e['actual']) > 0.1]
n_direction_match = sum(1 for e in detectable
                        if (e['actual'] > 0 and e['predicted'] > 0) or
                           (e['actual'] < 0 and e['predicted'] < 0))

print(f"Detectable proteins: {len(detectable)}")
print(f"Direction agreement: {n_direction_match}/{len(detectable)}")
print(f"\nTop 6 by effect size:")
for i, e in enumerate(expr_effects[:6]):
    print(f"  {i+1}. {e['label']:<30s} actual={e['actual']:+.4f}  pred={e['predicted']:+.4f}")

# ============================================================
# Figure 3A: cd52_panel_a.png
# ============================================================
fig_a, ax_a = plt.subplots(1, 1, figsize=(7, 5), dpi=300)

# Panel A: RNA scatter
ax = ax_a
ax.scatter(mean_rna_target, mean_rna_pred, alpha=0.15, s=3, c='#64B5F6', rasterized=True)
ax.plot([-0.5, 0.5], [-0.5, 0.5], 'k--', alpha=0.4, linewidth=1)
if CD52_CDT_IDX is not None:
    ax.scatter(mean_rna_target[CD52_CDT_IDX], mean_rna_pred[CD52_CDT_IDX],
               c='#D32F2F', s=80, zorder=5, edgecolors='white', linewidths=1,
               label=f'CD52 (self-KD)')
    ax.legend(fontsize=9, loc='upper left')
ax.set_xlabel('Actual RNA log2FC (pseudo-bulk)', fontsize=10)
ax.set_ylabel('Predicted RNA log2FC', fontsize=10)
ax.set_title('A', fontsize=14, fontweight='bold', loc='left')
ax.text(0.95, 0.05, f'Per-gene r = {rna_r:.3f}',
        transform=ax.transAxes, fontsize=10, ha='right', va='bottom',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.7))

fig_a.tight_layout()
fig_a.savefig(FIGURES_DIR / 'cd52_panel_a.png', dpi=300, bbox_inches='tight', facecolor='white')
fig_a.savefig(LOCAL_FIG / 'cd52_panel_a.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f"Saved: cd52_panel_a.png")

# ============================================================
# Figure 3B: cd52_panel_b.png
# ============================================================
fig_b, ax_b = plt.subplots(1, 1, figsize=(7, 5), dpi=300)

# Panel B: Protein scatter (expressed only)
ax = ax_b
expr_pred = mean_prot_pred[expressed_mask]
expr_target = mean_prot_target[expressed_mask]

ax.scatter(expr_target, expr_pred, alpha=0.7, s=40, c='#FF9800',
           edgecolors='white', linewidths=0.5)
ax.plot([-0.5, 0.5], [-0.5, 0.5], 'k--', alpha=0.4, linewidth=1)

# Label top 6 changed proteins with proper names
top_changed = np.argsort(np.abs(expr_target))[::-1][:6]
for idx in top_changed:
    prot_i = expressed_indices[idx]
    lbl = get_protein_label(protein_names[prot_i])
    ax.annotate(lbl, (expr_target[idx], expr_pred[idx]),
                fontsize=7, fontweight='bold',
                xytext=(5, 5), textcoords='offset points',
                arrowprops=dict(arrowstyle='-', alpha=0.3, linewidth=0.5))

ax.set_xlabel('Actual Protein Effect (pseudo-bulk DSB)', fontsize=10)
ax.set_ylabel('Predicted Protein Effect', fontsize=10)
ax.set_title('B', fontsize=14, fontweight='bold', loc='left')
ax.text(0.95, 0.05, f'r = {prot_r:.3f}\n(65 expressed proteins)',
        transform=ax.transAxes, fontsize=10, ha='right', va='bottom',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.7))

fig_b.tight_layout()
fig_b.savefig(FIGURES_DIR / 'cd52_panel_b.png', dpi=300, bbox_inches='tight', facecolor='white')
fig_b.savefig(LOCAL_FIG / 'cd52_panel_b.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f"Saved: cd52_panel_b.png")

# ============================================================
# Figure 3C: cd52_side_effect_map.png (0.45\textwidth, Panel C only)
# ============================================================
fig_c, ax_c = plt.subplots(1, 1, figsize=(7, 7), dpi=300)

# Panel C: Side effect map — 25 detectable proteins
ax = ax_c
n_show = min(25, len(detectable))
labels_c = [e['label'] for e in detectable[:n_show]]
actual_c = [e['actual'] for e in detectable[:n_show]]
pred_c = [e['predicted'] for e in detectable[:n_show]]

# Clinical relevance color coding
clinical_genes = {'TFRC', 'CD81', 'MCAM', 'CD58', 'ICAM1', 'FCGR2A', 'TNFRSF13C', 'CD40'}

bar_colors = []
for e in detectable[:n_show]:
    gene = e['gene']
    if gene in clinical_genes:
        bar_colors.append('#E53935')
    elif e['actual'] > 0:
        bar_colors.append('#42A5F5')
    else:
        bar_colors.append('#90CAF9')

y_pos = np.arange(n_show)
bar_h = 0.35

# Measured (solid bars)
ax.barh(y_pos + bar_h/2, actual_c, bar_h, color=bar_colors, alpha=0.8,
        label='Measured', edgecolor='white', linewidth=0.5)
# Predicted (hatched bars)
ax.barh(y_pos - bar_h/2, pred_c, bar_h, color=bar_colors, alpha=0.4,
        label='Predicted', edgecolor='white', linewidth=0.5, hatch='///')

ax.set_yticks(y_pos)
ax.set_yticklabels(labels_c, fontsize=8)
ax.invert_yaxis()
ax.axvline(0, color='black', linewidth=0.5)
ax.set_xlabel('Protein Effect (DSB difference)', fontsize=10)
ax.set_title('C', fontsize=14, fontweight='bold', loc='left')

# Direction agreement annotation
ax.text(0.97, 0.55, f'{n_direction_match}/{len(detectable)}\ndirection\nagreement',
        transform=ax.transAxes, fontsize=10, ha='right', va='top',
        fontweight='bold', color='#2E7D32',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#C8E6C9', alpha=0.7))

# Legend
legend_elements = [
    mpatches.Patch(facecolor='#E53935', alpha=0.8, label='Clinically relevant'),
    mpatches.Patch(facecolor='#42A5F5', alpha=0.8, label='Measured (solid)'),
    mpatches.Patch(facecolor='#42A5F5', alpha=0.4, hatch='///', label='Predicted (hatched)'),
]
ax.legend(handles=legend_elements, fontsize=8, loc='lower right')

fig_c.tight_layout()
fig_c.savefig(FIGURES_DIR / 'cd52_panel_c.png', dpi=300, bbox_inches='tight', facecolor='white')
fig_c.savefig(LOCAL_FIG / 'cd52_panel_c.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f"Saved: cd52_panel_c.png")

In [ ]:
# ============================================================
# Diagnostic: verify ADT mapping and figure outputs
# ============================================================

print("Side effect figures generated in Cell 9.")
print(f"  cd52_kd_predictions.png  — Panels A (RNA), B (Protein)")
print(f"  cd52_side_effect_map.png — Panel C (25 detectable proteins)")
print(f"\nTop 6 proteins by measured effect:")
for i, e in enumerate(expr_effects[:6]):
    direction = '↑' if e['actual'] > 0 else '↓'
    match = '✓' if (e['actual'] > 0 and e['predicted'] > 0) or (e['actual'] < 0 and e['predicted'] < 0) else '✗'
    print(f"  {i+1}. {e['label']:<30s} actual={e['actual']:+.4f}  pred={e['predicted']:+.4f}  {match} {direction}")

In [ ]:
# ============================================================
# E. Resistance Prediction: mRNA↓ but Protein maintained
# ============================================================

# For each protein that has a matching RNA gene:
# Compare RNA change vs Protein change
# Divergence = potential post-translational compensation = resistance

resistance_data = []
for i in expressed_indices:
    gene = adt_to_gene.get(protein_names[i])
    if gene and gene in gene_to_cdt_idx:
        rna_idx = gene_to_cdt_idx[gene]
        rna_change = mean_rna_target[rna_idx]
        prot_change = mean_prot_target[i]

        # Resistance indicator: RNA changes but protein doesn't follow
        if abs(rna_change) > 0.01:  # meaningful RNA change
            ratio = abs(prot_change) / (abs(rna_change) + 1e-8)
            divergence = abs(rna_change) - abs(prot_change)
            resistance_data.append({
                'protein_id': protein_names[i],
                'protein': adt_to_protein.get(protein_names[i], protein_names[i]),
                'gene': gene,
                'rna_change': rna_change,
                'prot_change': prot_change,
                'prot_rna_ratio': ratio,
                'divergence': divergence,
            })

if len(resistance_data) == 0:
    print("No resistance candidates found (no expressed proteins with matching RNA gene and |RNA change| > 0.01).")
else:
    res_df = pd.DataFrame(resistance_data).sort_values('divergence', ascending=False)

    print("Resistance Candidates (RNA changes, protein maintained):")
    print(res_df.head(15).to_string(index=False))

    # Visualization
    if len(res_df) > 3:
        fig, ax = plt.subplots(figsize=(10, 8))
        ax.scatter(res_df['rna_change'], res_df['prot_change'], s=50, c='purple', alpha=0.7)
        ax.plot([-0.3, 0.3], [-0.3, 0.3], 'k--', alpha=0.3, label='Expected (1:1)')
        ax.axhline(0, color='gray', alpha=0.3)
        ax.axvline(0, color='gray', alpha=0.3)

        # Label outliers (high divergence)
        for _, row in res_df.head(10).iterrows():
            ax.annotate(row['gene'], (row['rna_change'], row['prot_change']),
                        fontsize=8, fontweight='bold')

        ax.set_xlabel('RNA log2FC (measured)')
        ax.set_ylabel('Protein effect (measured)')
        ax.set_title('CD52 KD: RNA vs Protein Changes per Gene\n'
                     'Off-diagonal = post-translational regulation = resistance', fontsize=13)
        ax.legend()
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / 'cd52_resistance_rna_vs_protein.png', dpi=200, bbox_inches='tight')
        plt.show()

In [ ]:
# ============================================================
# F. Signal Propagation: DNA → RNA → Protein
# ============================================================
# Extract attention at each stage to trace information flow

ci = cd52_cells[0]
enf_idx = target_to_enformer[CD52_TARGET_IDX]
dna_t = torch.from_numpy(enformer_emb[enf_idx:enf_idx+1].astype(np.float32)).to(device)
rna_t = torch.from_numpy(cell_expr[ci:ci+1].astype(np.float32)).to(device)
prot_t = torch.from_numpy(protein_dsb[ci:ci+1].astype(np.float32)).to(device)

with torch.no_grad():
    _, _, attn_maps = model(dna_t, rna_t, prot_t, return_attention=True)

print("Attention maps extracted:")
for key, val in attn_maps.items():
    print(f"  {key}: {val.shape}")

# Signal flow figure
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Stage 1: DNA Self-Attention (CD52 locus)
dna_self = attn_maps['dna_self_attn_1'].numpy().squeeze(0).mean(axis=0)  # [896, 896]
axes[0].imshow(dna_self, cmap='hot', aspect='auto',
               vmax=np.percentile(dna_self, 99))
axes[0].set_title('Stage 1: DNA Self-Attention\n[896×896]', fontsize=12)
axes[0].set_xlabel('Key position')
axes[0].set_ylabel('Query position')

# Stage 2: DNA→RNA Cross-Attention (CD52 gene row)
cross_dr = attn_maps['dna_to_rna_cross'].numpy().squeeze(0).mean(axis=0)  # [2361, 896]
if CD52_CDT_IDX is not None:
    cd52_dna_attn = cross_dr[CD52_CDT_IDX, :]  # [896]
    axes[1].plot(cd52_dna_attn, 'b-', linewidth=0.8)
    axes[1].fill_between(range(len(cd52_dna_attn)), cd52_dna_attn, alpha=0.3)
    axes[1].axvline(448, color='red', linestyle='--', label='TSS')
    axes[1].set_title('Stage 2: DNA→RNA Cross-Attention\nCD52 gene', fontsize=12)
    axes[1].set_xlabel('DNA position (896 bins)')
    axes[1].set_ylabel('Attention weight')
    axes[1].legend()

# Stage 3: RNA→Protein Cross-Attention
cross_rp = attn_maps['rna_to_protein_cross'].numpy().squeeze(0).mean(axis=0)  # [189, 2361]
# Show top RNA genes for each protein (subset)
n_show_prot = min(30, n_expressed)
show_prot_idx = expressed_indices[:n_show_prot]
cross_sub = cross_rp[show_prot_idx, :]
# Show only top 50 RNA genes by mean attention
top_rna_for_cross = np.argsort(cross_sub.mean(axis=0))[::-1][:50]
cross_sub2 = cross_sub[:, top_rna_for_cross]
axes[2].imshow(cross_sub2, cmap='magma', aspect='auto')
axes[2].set_title('Stage 3: RNA→Protein Cross-Attention\n★CDT-III unique', fontsize=12)
axes[2].set_xlabel('Top 50 RNA genes')
axes[2].set_ylabel('Expressed proteins')

plt.suptitle('CD52: Information Flow DNA → RNA → Protein',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'cd52_signal_flow.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# G. VCE-T vs VCE-P Embedding Comparison
# ============================================================
# Extract intermediate embeddings: cell_emb_rna (VCE-T) vs cell_emb_protein (VCE-P)

def extract_cell_embeddings(model, cell_indices, enformer_emb, cell_expr,
                             protein_dsb, target_gene_idx, target_to_enformer,
                             device, max_cells=200):
    """Extract VCE-T and VCE-P cell embeddings."""
    emb_rna_list, emb_prot_list = [], []
    labels = []

    for ci in cell_indices[:max_cells]:
        enf_idx = target_to_enformer[target_gene_idx[ci]]
        dna_t = torch.from_numpy(enformer_emb[enf_idx:enf_idx+1].astype(np.float32)).to(device)
        rna_t = torch.from_numpy(cell_expr[ci:ci+1].astype(np.float32)).to(device)
        prot_t = torch.from_numpy(protein_dsb[ci:ci+1].astype(np.float32)).to(device)

        with torch.no_grad():
            # Manual forward to get intermediate embeddings
            dna = model.dna_projector(dna_t)
            rna = model.rna_encoder(rna_t)
            protein = model.protein_encoder(prot_t)

            for layer in model.dna_self_attn_layers:
                dna = layer(dna)
            for layer in model.rna_self_attn_layers:
                rna = layer(rna)
            for layer in model.protein_self_attn_layers:
                protein = layer(protein)

            rna = model.dna_to_rna(query=rna, key_value=dna)
            protein = model.rna_to_protein(query=protein, key_value=rna)

            cell_emb_rna = model.vce_t(dna, rna)
            cell_emb_protein = model.vce_p(cell_emb_rna, protein)

        emb_rna_list.append(cell_emb_rna.cpu().numpy().flatten())
        emb_prot_list.append(cell_emb_protein.cpu().numpy().flatten())

    return np.array(emb_rna_list), np.array(emb_prot_list)


# Collect embeddings for val gene cells
all_embed_cells = []
all_embed_labels = []

for gene_name in ['CD52'] + [g for g in val_genes if g != 'CD52']:
    g_idx = gene_name_to_target_idx.get(gene_name)
    if g_idx is None or g_idx not in target_to_enformer:
        continue
    g_cells = [i for i in range(n_cells)
               if target_gene_idx[i] == g_idx and protein_matched_mask[i]][:100]
    all_embed_cells.extend(g_cells)
    all_embed_labels.extend([gene_name] * len(g_cells))

if len(all_embed_cells) > 50:
    emb_rna, emb_prot = extract_cell_embeddings(
        model, all_embed_cells, enformer_emb, cell_expr, protein_dsb,
        target_gene_idx, target_to_enformer, device, max_cells=len(all_embed_cells))

    # t-SNE
    tsne_rna = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(emb_rna)
    tsne_prot = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(emb_prot)

    labels_arr = np.array(all_embed_labels)
    unique_labels = list(set(all_embed_labels))
    cmap = plt.cm.Set1(np.linspace(0, 1, len(unique_labels)))

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    for li, label in enumerate(unique_labels):
        mask = labels_arr == label
        ms = 20 if label == 'CD52' else 8
        axes[0].scatter(tsne_rna[mask, 0], tsne_rna[mask, 1],
                        c=[cmap[li]], s=ms, alpha=0.6, label=label)
        axes[1].scatter(tsne_prot[mask, 0], tsne_prot[mask, 1],
                        c=[cmap[li]], s=ms, alpha=0.6, label=label)

    axes[0].set_title('VCE-T: cell_emb_rna (DNA+RNA)', fontsize=13)
    axes[0].legend(fontsize=8)
    axes[1].set_title('VCE-P: cell_emb_protein (RNA+Protein) ★NEW', fontsize=13)
    axes[1].legend(fontsize=8)

    plt.suptitle('VCE-T vs VCE-P Cell Embeddings (t-SNE)', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'vce_t_vs_vce_p_tsne.png', dpi=200, bbox_inches='tight')
    plt.show()
else:
    print("Not enough cells for embedding analysis.")

In [ ]:
# ============================================================
# Save Results
# ============================================================

# Prediction CSV
pred_df = pd.DataFrame({
    'protein_id': protein_names,
    'protein_name': [adt_to_protein.get(p, p) for p in protein_names],
    'gene_symbol': [adt_to_gene.get(p, '') for p in protein_names],
    'expressed': expressed_mask,
    'actual_effect': mean_prot_target,
    'predicted_effect': mean_prot_pred,
})
pred_df.to_csv(OUTPUT_BASE / 'cd52_protein_predictions.csv', index=False)

# Resistance data
if len(resistance_data) > 0:
    res_df.to_csv(OUTPUT_BASE / 'cd52_resistance_analysis.csv', index=False)

print("Saved:")
print(f"  {OUTPUT_BASE / 'cd52_protein_predictions.csv'}")
print(f"  {OUTPUT_BASE / 'cd52_resistance_analysis.csv'}")
print(f"  Figures in {FIGURES_DIR}")
print("\nNB3 (CD52/Alemtuzumab case study) complete.")